In [13]:
from langchain_groq import ChatGroq                          # replaces ChatOpenAI
from langchain_community.document_loaders import PyPDFLoader
from langchain_experimental.text_splitter import SemanticChunker
from dotenv import load_dotenv
import os
load_dotenv()

llm = ChatGroq(model="llama-3.3-70b-versatile")  # free on Groq


from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"   # free, fast, runs locally
)

c:\Users\Dhruv\OneDrive\Desktop\ai fundamentals class\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Dhruv\OneDrive\Desktop\ai fundamentals class\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Dhruv\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mod

In [14]:
text_data = PyPDFLoader("../LLM/NovaS.pdf").load()
full_text = "\n".join([page.page_content for page in text_data])
full_text

'NovaSphere Technologies is a fictional organization created to represent a modern data \nand technology company that has grown gradually over the years. The organization was \nfounded in 2016 by a small group of software engineers who strongly believed that data \nwould become one of the most valuable assets for every business in the future. At the \nbeginning, the company did not have large investments or a big office. Instead, it started \nwith only six employees working together in a small shared workspace. The founders were \nnot focused on becoming successful overnight. Their main goal was to build strong \ntechnical knowledge, gain practical experience, and slowly grow by delivering real value to \ntheir clients. Most of the early work involved helping small companies understand their \nexisting data and use simple reporting solutions to make better business decisions. \nDuring the first year of operations, the company worked mostly with local startups that did \nnot have large 

In [20]:
chunker = SemanticChunker(
    HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2"),  
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=60
)

semantic_chunks = chunker.create_documents([full_text])
print(semantic_chunks)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3075.90it/s]


[Document(metadata={}, page_content='NovaSphere Technologies is a fictional organization created to represent a modern data \nand technology company that has grown gradually over the years. The organization was \nfounded in 2016 by a small group of software engineers who strongly believed that data \nwould become one of the most valuable assets for every business in the future.'), Document(metadata={}, page_content='At the \nbeginning, the company did not have large investments or a big office.'), Document(metadata={}, page_content='Instead, it started \nwith only six employees working together in a small shared workspace. The founders were \nnot focused on becoming successful overnight.'), Document(metadata={}, page_content='Their main goal was to build strong \ntechnical knowledge, gain practical experience, and slowly grow by delivering real value to \ntheir clients. Most of the early work involved helping small companies understand their \nexisting data and use simple reporting sol

In [21]:
from langchain_community.vectorstores import Chroma

In [22]:
embed_model = HuggingFaceEmbeddings(model="all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3372.97it/s]


In [23]:
chroma_db = Chroma.from_documents(semantic_chunks, embed_model, persist_directory="./chroma_db_semantic")

In [24]:
chroma_db_con = Chroma(persist_directory="./chroma_db_semantic", embedding_function=embed_model)

C:\Users\Dhruv\AppData\Local\Temp\ipykernel_30892\2430328870.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  chroma_db_con = Chroma(persist_directory="./chroma_db_semantic", embedding_function=embed_model)


In [25]:
chroma_db_con.similarity_search("By 2019, how many employees were there?", k=3)

[Document(metadata={}, page_content='The number of employees \nincreased to more than fifty, and the company started working with larger clients from \ndifferent industries.'),
 Document(metadata={}, page_content='Instead, it started \nwith only six employees working together in a small shared workspace. The founders were \nnot focused on becoming successful overnight.'),
 Document(metadata={}, page_content='The company also started focusing more \non improving the quality of its work instead of simply increasing the number of projects. The founders believed that long-term growth would come only if the organization built a \nstrong reputation for reliability and consistency. Because of this mindset, the team spent \nextra time testing their solutions and making sure that the final product worked smoothly \nfor the client. By the beginning of 2019, the organization had grown to more than fifteen employees.')]

In [26]:
llm = ChatGroq(model="llama-3.3-70b-versatile")

In [30]:
user_query = input("Enter your question: ")

rel_chunks = chroma_db_con.similarity_search(user_query, k=3)

rel_chunks_content = []
for i, chunk in enumerate(rel_chunks):
    rel_chunks_content.append(chunk.page_content)
rel_chunks_content = str(rel_chunks_content)

llm.invoke(f"{user_query}, Use the following context to answer the question: {rel_chunks_content}")

AIMessage(content='Based on the provided context, the total number of employees at two different points in time is mentioned:\n\n1. The company started with 6 employees.\n2. By the beginning of 2019, the organization had grown to more than 15 employees.\n3. Later, the number of employees increased to more than 50.\n\nThe most recent and highest number mentioned is "more than 50". Therefore, the total number of employees now is more than 50. The exact number is not specified.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 102, 'prompt_tokens': 208, 'total_tokens': 310, 'completion_time': 0.354964316, 'completion_tokens_details': None, 'prompt_time': 0.010813521, 'prompt_tokens_details': None, 'queue_time': 0.047500004, 'total_time': 0.365777837}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_ce7bc1685b', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e45d5-5f23-7722-8